In [2]:
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [3]:
BATCH_SIZE = 32
IMG_SIZE = (224,224)

train_ds = tf.keras.utils.image_dataset_from_directory(
    "E:\datasets\PlantVillage",
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "E:\datasets\PlantVillage",
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

Found 20638 files belonging to 15 classes.
Using 16511 files for training.
Found 20638 files belonging to 15 classes.
Using 4127 files for validation.


In [4]:
class_names = train_ds.class_names
print(class_names)
print(len(class_names))

['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']
15


In [5]:
for X_batch, y_batch in train_ds.take(1):
    print(X_batch.shape)
    print(y_batch.shape)

(32, 224, 224, 3)
(32,)


In [6]:
AUTOTUNE= tf.data.AUTOTUNE
train_ds= train_ds.prefetch(AUTOTUNE)
val_ds= val_ds.prefetch(AUTOTUNE)

In [7]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
])

In [8]:
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

In [9]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape = IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet')

In [11]:
NUM_CLASSES = 15

In [12]:
base_model.trainable= False

In [13]:
inputs = tf.keras.Input(shape=(224,224,3))
x= data_augmentation(inputs)
x= preprocess_input(x)
x= base_model(x, training= False)
x= tf.keras.layers.GlobalAveragePooling2D()(x)
x= tf.keras.layers.Dropout(0.2)(x)
outputs= tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)
model= tf.keras.Model(inputs, outputs)

In [14]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

In [16]:
early_stop = EarlyStopping(
    monitor='val_loss',       # what to watch (val_loss or val_accuracy)
    patience=3,               # number of epochs with no improvement before stopping
    restore_best_weights=True # roll back to the best weights
)

checkpoint = ModelCheckpoint(
    filepath='best_model.h5', # file to save the best model
    monitor='val_loss',       # metric to monitor
    save_best_only=True,      # only save when val_loss improves
    verbose=1
)

In [17]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 509s 987ms/step - accuracy: 0.8724 - loss: 0.3872 - val_accuracy: 0.8902 - val_loss: 0.3389
Epoch 2/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 601s 1s/step - accuracy: 0.8857 - loss: 0.3491 - val_accuracy: 0.9089 - val_loss: 0.2909
Epoch 3/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 605s 1s/step - accuracy: 0.8970 - loss: 0.3144 - val_accuracy: 0.9096 - val_loss: 0.2759
Epoch 4/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 577s 1s/step - accuracy: 0.8998 - loss: 0.2987 - val_accuracy: 0.9087 - val_loss: 0.2669
Epoch 5/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 578s 1s/step - accuracy: 0.9022 - loss: 0.2899 - val_accuracy: 0.9130 - val_loss: 0.2623
Epoch 6/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 556s 1s/step - accuracy: 0.9073 - loss: 0.2753 - val_accuracy: 0.9147 - val_loss: 0.2549
Epoch 7/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 568s 1s/step - accuracy: 0.9072 - loss: 0.2746 - val_accuracy: 0.9108 - val_loss: 0.2536
Epoch 8/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 608s 1s/step - accuracy: 0.9097 - loss: 0.2633 - val_a